[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/06_strukturierte_ausgabe.ipynb)

# Sitzung 6 — Strukturierte Ausgabe

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI · 11.11.2026**

Ein Dashboard, eine Datenbank, der nächste Schritt in der Pipeline: Sie alle brauchen **Felder**, keinen Fließtext. Bisher haben wir die Antworten des Modells mit regulären Ausdrücken herausgefischt und gehofft. Heute machen wir das verlässlich.

**Ablauf:**

1. **Was alles schiefgeht (Abschnitt 1):** elf typische Antworten, und warum `json.loads` allein nicht reicht.
2. **Ein Schema als Vertrag (Abschnitt 2–3):** genau festlegen, wie eine gültige Antwort aussieht, prüfen, und bei Fehlern das Modell selbst korrigieren lassen.
3. **Schema direkt beim Anbieter (Abschnitt 4):** das Format vom Anbieter durchsetzen lassen, und was trotzdem schiefgehen kann.
4. **Embeddings, Teil 2 (Abschnitt 5):** Gruppen finden, ohne Katalog, und prüfen, ob unser Katalog passt.

### Rückblick Sitzung 5

```text
1. "Ein Drittel negativ" hilft     -> Sentiment pro Thema: worüber wird geklagt?
   niemandem
2. Themen frei benennen lassen     -> fester Katalog mit Definitionen, sonst kann man nicht zählen
3. Stichwort-Regeln                -> gut auf bekannten Texten, brechen bei neuen Formulierungen
4. LLM                             -> Themen + Stimmung in einem Schritt, aber messen!
5. Baustellen-Liste                -> zwei Rankings: am häufigsten vs. am schlechtesten bewertet
6. Embeddings                      -> Bedeutung als Zahlen: ähnlich = nah; findet das Thema, kaum die Stimmung
```

Offen geblieben: Unser Parser hat Einträge **stillschweigend verworfen**, wenn das Modell vom Format abwich. Heute schauen wir genau hin.

## 0. Setup

**Was passiert hier?** Wie immer `radar.py` laden und den bereinigten Datensatz holen. Neu ist das Paket **pydantic**: eine Standard-Bibliothek in Python, mit der man beschreibt, wie Daten aussehen müssen, und sie automatisch prüft. In Colab ist sie schon installiert.

In [ ]:
import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # vor allen Imports setzen
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TQDM_DISABLE"] = "1"
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")

!pip install anthropic sentence-transformers --quiet

import json, re, time, math, random, urllib.request, importlib
from collections import Counter
from typing import Literal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown
from pydantic import BaseModel, Field, ValidationError
pd.set_option("display.max_colwidth", 90)

URL = "https://raw.githubusercontent.com/kevisback/bda-course/session-06/wise2627/notebooks/radar.py"
try:
    urllib.request.urlretrieve(URL, "radar.py")
except Exception as e:
    print("Download fehlgeschlagen, nutze lokale radar.py:", e)
import radar
importlib.reload(radar)
radar.setup(backend="anthropic", model="claude-haiku-4-5")

_ = radar.embed(["Test"])      # Embedding-Modell vorab laden (einmalig ca. 1 Minute)

df, wahrheit = radar.lade_sauber()
THEMEN = radar.THEMEN
print(df.shape)

## 1. Was alles schiefgehen kann

Unser Ziel für heute: Aus einer **frei geschriebenen Bewertung** soll das Modell **feste Felder** machen, die ein Programm direkt weiterverarbeiten kann.

```text
Bewertung (Freitext)                                   Antwort (JSON-Objekt)
"Der Klang ist wirklich super, aber die App     ->     {"sentiment": "mixed",
 stürzt ständig ab."                                    "themen": [{"thema": "Klang", "polaritaet": "positiv"},
                                                                   {"thema": "App & Bedienung", "polaritaet": "negativ"}],
                                                        "begruendung": "Lob für Klang, Kritik an der App."}
```

Die drei Felder:
- **sentiment:** die Gesamtstimmung, nur eines von vier Labels (`positive`, `negative`, `mixed`, `neutral`), wie in Sitzung 2 und 3.
- **themen:** eine Liste der angesprochenen Themen aus unserem Katalog, jedes mit Polarität, wie in Sitzung 5.
- **begruendung:** ein kurzer Satz, damit ein Mensch nachvollziehen kann, warum das Modell so entschieden hat.

Aus Sprache werden Felder: Statt „irgendwo im Text steht, dass die App nervt" hat das Programm jetzt `thema = "App & Bedienung"` und `polaritaet = "negativ"`, und das kann man zählen, filtern und in ein Dashboard schreiben.

Ein Modell ist aber ein **Textgenerator**. Es gibt keine Garantie, dass genau das herauskommt. Hier sind elf Antworten auf diese Bewertung, wie sie in der Praxis vorkommen. **Was passiert in der nächsten Zelle?** Wir legen die elf Beispiele als Liste an und zeigen sie.

In [ ]:
BEWERTUNG = "Der Klang ist wirklich super, aber die App stürzt ständig ab."

KORREKT = '{"sentiment": "mixed", "themen": [{"thema": "Klang", "polaritaet": "positiv"}, {"thema": "App & Bedienung", "polaritaet": "negativ"}], "begruendung": "Lob für Klang, Kritik an der App."}'

ANTWORTEN = {
    "1 sauber":                 KORREKT,
    "2 Text davor":             "Gerne! Hier ist die Analyse:\n" + KORREKT,
    "3 Code-Block":             "```json\n" + KORREKT + "\n```",
    "4 Komma zu viel":          KORREKT.replace('"negativ"}]', '"negativ"},]'),
    "5 Feldname falsch":        KORREKT.replace('"sentiment"', '"Sentiment"'),
    "6 Label erfunden":         KORREKT.replace('"mixed"', '"gemischt"'),
    "7 Thema nicht im Katalog": KORREKT.replace('"Klang"', '"Klangqualität"'),
    "8 Feld fehlt":             '{"sentiment": "mixed", "begruendung": "Lob und Kritik."}',
    "9 falscher Typ":           KORREKT.replace('"themen": [', '"themen": "Klang, App", "x": ['),
    "10 abgeschnitten":         KORREKT[:90],
    "11 Begründung widerspricht": KORREKT.replace('"mixed"', '"positive"'),
}
WAS_ANDERS = {
    "1 sauber":                 "nichts, so soll es sein",
    "2 Text davor":             "Höflichkeitssatz vor dem JSON",
    "3 Code-Block":             "JSON in ```json ... ``` eingepackt",
    "4 Komma zu viel":          "Komma vor der schließenden ]",
    "5 Feldname falsch":        "„Sentiment\" statt „sentiment\"",
    "6 Label erfunden":         "„gemischt\" statt „mixed\"",
    "7 Thema nicht im Katalog": "„Klangqualität\" statt „Klang\"",
    "8 Feld fehlt":             "„themen\" fehlt ganz",
    "9 falscher Typ":           "„themen\" ist Text statt Liste",
    "10 abgeschnitten":         "bricht mitten im JSON ab (max_tokens)",
    "11 Begründung widerspricht": "„positive\", aber Begründung nennt Kritik",
}

print("Bewertung:", BEWERTUNG, "\n")
tabelle = pd.DataFrame({"Was ist anders?": WAS_ANDERS,
                        "Antwort des Modells (vollständig, ⏎ = Zeilenumbruch)":
                            {k: v.replace("\n", " ⏎ ") for k, v in ANTWORTEN.items()}})
with pd.option_context("display.max_colwidth", None):
    display(tabelle)

**Wie lese ich das?** Fall 1 ist, was wir wollen. Die anderen zehn sind typische Abweichungen, in drei Arten:

- **2–4** sind *Formatfehler*: Der Inhalt stimmt, aber drumherum steht etwas, oder ein Zeichen ist zu viel.
- **5–9** sind *Inhaltsfehler*: Das JSON ist technisch gültig, passt aber nicht zu unserer Erwartung (falscher Name, erfundenes Label, fehlendes Feld, Text statt Liste).
- **10** ist abgeschnitten, weil `max_tokens` erreicht war (Sitzung 2: `stop_reason`).
- **11** ist ein *Widerspruch*: Format und Felder stimmen, aber das Label `positive` passt nicht zur Begründung „Kritik an der App". Welche der beiden Angaben stimmt? Das verrät nur die Begründung.

Merkt euch Fall 11. Wir schauen gleich, welche Prüfung ihn findet.

**Was passiert in der nächsten Zelle?** Wir probieren den naheliegendsten Weg: `json.loads`, die Standardfunktion zum Einlesen von JSON.

In [ ]:
for name, roh in ANTWORTEN.items():
    try:                                   # ohne try würde die erste kaputte Antwort alles abbrechen
        json.loads(roh)
        print(f"✅ {name:26} lässt sich einlesen")
    except json.JSONDecodeError as e:
        print(f"❌ {name:26} {e.msg}")

**Wie lese ich das?** Zwei Arten von Problemen werden sichtbar:

1. Bei 2, 3, 4 und 10 **scheitert** `json.loads`. Hier fängt `try/except` das ab; ohne diesen Schutz würde eine Schleife über 300 Bewertungen bei der ersten kaputten Antwort abbrechen (Sitzung 3). Die Fehlermeldungen sind technisch und auf Englisch, sie sagen nur, *wo* das Einlesen gescheitert ist, nicht *was* gemeint war.
2. Viel gefährlicher: Bei 5 bis 9 und bei 11 meldet `json.loads` **Erfolg**. Das JSON ist ja gültig. Dass „gemischt" kein erlaubtes Label ist, die Themen fehlen oder Label und Begründung sich widersprechen, merkt es nicht. Diese Fehler laufen still in die Kennzahlen.

> **Nicht:** „Lässt sich einlesen, also stimmt es."
> **Sondern:** Gültiges JSON heißt nur: Die Klammern stimmen. Ob der **Inhalt** passt, muss man extra prüfen.

## 2. Ein Schema als Vertrag

Ein **Schema** beschreibt genau, wie eine gültige Antwort aussieht: welche Felder, welche Typen, welche Werte erlaubt sind. Es ist ein Vertrag zwischen dem Modell und unserem Programm.

Mit pydantic schreibt man das Schema als Python-Klasse. **Was passiert in der nächsten Zelle?** Wir legen zwei Klassen an:
- `Nennung`: ein Thema mit Polarität. `Literal[...]` heißt: nur genau diese Werte sind erlaubt.
- `Analyse`: die ganze Antwort, mit Sentiment, einer Liste von Nennungen und einer kurzen Begründung.

In [ ]:
class Nennung(BaseModel):
    thema: Literal[tuple(THEMEN)]                      # nur Themen aus dem Katalog
    polaritaet: Literal["positiv", "negativ"]

class Analyse(BaseModel):
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    themen: list[Nennung]                              # eine Liste, gern auch leer
    begruendung: str = Field(max_length=200)           # Text, höchstens 200 Zeichen

# So sieht ein gültiges Objekt aus:
Analyse(sentiment="mixed", themen=[{"thema": "Klang", "polaritaet": "positiv"}], begruendung="kurz")

Jetzt bauen wir den Prüfer in drei Stufen, jede fängt eine Art von Fehler:

```text
Rohantwort -> 1. JSON herausschneiden -> 2. einlesen (+ kleine Reparatur) -> 3. gegen Schema prüfen -> Analyse
                (Text davor, Code-Block)    (Komma zu viel)                  (Namen, Werte, Typen)
```

In [ ]:
def pruefe(roh):
    """-> (Analyse oder None, Fehlermeldung oder None)"""
    # Stufe 1: das JSON-Objekt aus dem Text herausschneiden
    m = re.search(r"\{.*\}", roh or "", re.DOTALL)
    if not m:
        return None, "Kein JSON-Objekt gefunden."
    text = re.sub(r",\s*([}\]])", r"\1", m.group(0))   # Stufe 2a: Komma vor schließender Klammer entfernen
    # Stufe 2: einlesen
    try:
        daten = json.loads(text)
    except json.JSONDecodeError as e:
        return None, f"Kein gültiges JSON: {e.msg}"
    # Stufe 3: gegen das Schema prüfen
    try:
        return Analyse.model_validate(daten), None
    except ValidationError as e:
        fehler = "; ".join(f"{'.'.join(str(x) for x in f['loc'])}: {f['msg']}" for f in e.errors())
        return None, fehler

**Was passiert jetzt?** Alle elf Antworten durch den Prüfer.

In [ ]:
def json_loads_ok(roh):
    try:
        json.loads(roh)
        return "✅"
    except json.JSONDecodeError:
        return "❌"

zeilen = []
for name, roh in ANTWORTEN.items():
    analyse, fehler = pruefe(roh)
    zeilen.append({"Fall": name,
                   "nur json.loads": json_loads_ok(roh),
                   "mit Prüfer": "✅" if analyse else "❌",
                   "Fehlermeldung des Prüfers": fehler or ""})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(zeilen).set_index("Fall"))

**Wie lese ich das?** Links das Ergebnis von vorhin, rechts mit unserem Prüfer. Spannend sind die Zeilen, in denen sich etwas ändert:

- **2, 3, 4:** vorher ❌, jetzt ✅. Stufe 1 schneidet den Text drumherum weg, Stufe 2a entfernt das überzählige Komma. Der Inhalt war ja richtig.
- **5 bis 9:** vorher ✅, jetzt ❌. Diese Fehler wären vorher still durchgerutscht. Jetzt werden sie **erkannt**, und die Fehlermeldung sagt genau, was nicht stimmt: welches Feld, welcher Wert, was erwartet war.
- **10** bleibt kaputt. Ein abgeschnittenes JSON kann man nicht sinnvoll reparieren.
- **11** ist in beiden Spalten ✅, und das ist das Problem. Jedes Feld hat einen erlaubten Wert, nur passen Label und Begründung nicht zusammen. Ein Schema prüft die **Form**, nicht den **Sinn**. Das merkt nur, wer die Begründung liest: ein Mensch bei Stichproben, oder ein zweiter Prüfschritt.

Die Fehlermeldungen sind nicht nur für uns nützlich. Im nächsten Abschnitt geben wir sie dem Modell zurück.

> 💡 **json.loads oder Prüfer?** `json.loads` prüft, ob der Text gültiges JSON ist (Klammern, Anführungszeichen). Der Prüfer prüft, ob die Antwort unserem **Vertrag** entspricht (Felder, Typen, erlaubte Werte) und liefert ein Objekt, auf das sich der Rest der Pipeline verlassen kann. Ob der Inhalt **stimmt**, prüft keiner von beiden (Fall 11).

### ✏️ Eure Aufgabe: ein Feld ergänzen (10 min)

Die Produktmanagerin möchte zusätzlich wissen: **Muss sich jemand um diese Bewertung kümmern?** (z. B. weil ein Defekt gemeldet wird).

1. Ergänzt in `Analyse` ein Feld `dringend` vom Typ `bool` (wahr/falsch).
2. Führt die Prüfung der elf Antworten erneut aus. Was passiert mit Fall 1, und warum?
3. Wie müsste man `KORREKT` ändern, damit Fall 1 wieder gültig ist?

In [ ]:
class Analyse(BaseModel):
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    themen: list[Nennung]
    begruendung: str = Field(max_length=200)
    # TODO: Feld "dringend" vom Typ bool

In [ ]:
# Lösung: erst selbst versuchen, dann ausführen und vergleichen.
class Analyse(BaseModel):
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    themen: list[Nennung]
    begruendung: str = Field(max_length=200)
    dringend: bool                                       # neu: Pflichtfeld, wahr/falsch

# pruefe() nutzt automatisch die neue Klasse Analyse, wir müssen es nicht neu schreiben.

# Aufgabe 2: alle Antworten erneut prüfen
zeilen = []
for name, roh in ANTWORTEN.items():
    analyse, fehler = pruefe(roh)
    zeilen.append({"Fall": name, "gültig mit neuem Schema": "✅" if analyse else "❌",
                   "Fehlermeldung": fehler or ""})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(zeilen).set_index("Fall"))

# Aufgabe 3: Fall 1 mit dem neuen Feld
KORREKT_NEU = KORREKT.replace('"begruendung"', '"dringend": false, "begruendung"')
print("Fall 1 angepasst:", KORREKT_NEU, "\n")
analyse, fehler = pruefe(KORREKT_NEU)
print("gültig:", "✅" if analyse else "❌", "| dringend =", analyse.dringend if analyse else "-")

**Wie lese ich das?**
- **Aufgabe 2:** Mit dem neuen Pflichtfeld fallen **alle** Antworten durch, auch Fall 1, der vorhin perfekt war. An den Antworten hat sich nichts geändert, nur am Vertrag. Die Fehlermeldung sagt genau, was fehlt: `dringend: Field required`.
- **Aufgabe 3:** Fügt man `"dringend": false` ein, ist Fall 1 wieder gültig.

Das ist gewollt: Ein Schema ist ein Vertrag, und wer ihn ändert, muss auch den **Prompt** anpassen. Sonst liefert das Modell weiter das alte Format, und jede einzelne Antwort fällt durch. Genau das machen wir im nächsten Abschnitt: Der neue Prompt fragt `dringend` mit ab.

> 💡 In der Praxis: Schema und Prompt **gemeinsam** ändern und versionieren, wie eine Schnittstelle zwischen zwei Programmen.

## 3. Das Modell sich selbst korrigieren lassen

Bisher: Ungültige Antwort → `invalid` → Zeile verloren (oder einzeln wiederholen und hoffen). Besser: Wir sagen dem Modell **was** falsch war und lassen es korrigieren. Die Fehlermeldung aus dem Schema ist dafür genau richtig.

```text
Bewertung -> Modell -> prüfen -- gültig --> fertig
                          |
                       ungültig
                          |
                          v
   Bewertung + alte Antwort + Fehlermeldung -> Modell -> prüfen -> ... (höchstens 2 Korrekturen)
```

> ⚠️ Führt vorher die **Lösungszelle** der Aufgabe aus (Klasse `Analyse` mit `dringend`). Alles Folgende braucht das neue Feld.

**Was passiert in der nächsten Zelle?** Der Prompt (mit dem neuen Feld `dringend`) und eine Funktion, die das Modell fragt. Ohne Schlüssel baut sie eine Mock-Antwort zusammen. Diesen Teil müsst ihr nicht lesen.

In [ ]:
ANALYSE_PROMPT = """Du analysierst eine Produktbewertung zu kabellosen Ohrhörern.

Antworte ausschließlich mit einem JSON-Objekt mit genau diesen Feldern:
- "sentiment": "positive", "negative", "mixed" oder "neutral"
- "themen": Liste von {"thema": <Thema>, "polaritaet": "positiv" oder "negativ"}
   Erlaubte Themen: """ + ", ".join(THEMEN) + """
- "begruendung": höchstens 20 Wörter
- "dringend": true, wenn ein Defekt oder ein Problem gemeldet wird, um das sich jemand kümmern sollte, sonst false"""

def baue_anfrage(text, korrektur=None):
    """Die Nachricht ans Modell. korrektur = (alte Antwort, Fehlermeldung) oder None."""
    user = f'Bewertung:\n"""{text}"""'
    if korrektur:
        alt, fehler = korrektur
        user += (f"\n\nDeine letzte Antwort war ungültig.\nAntwort: {alt}\nFehler: {fehler}\n"
                 "Antworte nur mit dem korrigierten JSON-Objekt.")
    return user

def frage_modell(text, korrektur=None):
    """Eine Anfrage ans Modell."""
    user = baue_anfrage(text, korrektur)
    if radar.BACKEND == "mock":                     # Mock: Antwort aus Stichwörtern zusammenbauen
        themen = radar._mock_themen(text)
        s = json.loads(radar.call_llm(user)["text"])["sentiment"]
        return json.dumps({"sentiment": s, "themen": themen, "begruendung": "Mock",
                           "dringend": any(t["polaritaet"] == "negativ" for t in themen)}, ensure_ascii=False)
    return radar.call_llm(user, system=ANALYSE_PROMPT, max_tokens=400)["text"]

Ein echtes Modell liefert meist auf Anhieb gültiges JSON, die Korrektur wäre dann kaum zu sehen. Damit ihr sie seht, schalten wir eine **Störung** dazwischen: Mit einer einstellbaren Wahrscheinlichkeit verfälschen wir die erste Antwort nach einem der Fehlermuster von oben. Die Korrektur selbst läuft dann wieder echt.

**Was passiert in der nächsten Zelle?** Die Funktion `analysiere` fragt, prüft, und schickt bei Fehlern die Fehlermeldung zurück, höchstens zweimal. Dabei schreibt sie ein **Protokoll**: Störung ja/nein, jede Antwort, jede Fehlermeldung, jede Korrektur-Anfrage.

In [ ]:
STOERUNGEN = {                                   # gleiche Fehlermuster wie in Abschnitt 1
    "Feldname falsch (Sentiment)": lambda r: r.replace('"sentiment"', '"Sentiment"'),
    "Label erfunden (gemischt)":   lambda r: re.sub(r'"(positive|negative|mixed|neutral)"', '"gemischt"', r, count=1),
    "Feldname falsch (themes)":    lambda r: r.replace('"themen"', '"themes"'),
    "abgeschnitten":               lambda r: r[: len(r) // 2],
}

def analysiere(text, stoerung=0.0, max_korrekturen=2):
    """-> (Analyse oder None, Anzahl Anfragen, Protokoll)"""
    protokoll = {"stoerung": None, "original": None, "schritte": []}
    roh = frage_modell(text)                                    # Anfrage 1
    if random.random() < stoerung:                              # Störung nur bei der ersten Antwort
        protokoll["stoerung"] = random.choice(list(STOERUNGEN))
        protokoll["original"] = roh
        roh = STOERUNGEN[protokoll["stoerung"]](roh)
    for anfrage in range(1, max_korrekturen + 2):
        analyse, fehler = pruefe(roh)
        schritt = {"antwort": roh, "fehler": fehler, "gesendet": None}
        protokoll["schritte"].append(schritt)
        if analyse:
            return analyse, anfrage, protokoll
        if anfrage <= max_korrekturen:                          # Korrektur-Anfrage
            schritt["gesendet"] = baue_anfrage(text, korrektur=(roh, fehler))
            roh = frage_modell(text, korrektur=(roh, fehler))
    return None, anfrage, protokoll

**Was passiert jetzt?** 20 Bewertungen, bei 30 % Störung. Die Ausgabe hat drei Teile:
1. **Überblick:** alle 20 Bewertungen, welche gestört wurde und wie, wie viele Anfragen nötig waren.
2. **Bilanz:** Wie viele Anfragen hat das insgesamt gekostet?
3. **Ein Fall im Detail:** was das Modell geantwortet hat, was wir daraus gemacht haben, was wir ihm zurückgeschickt haben, und wie es korrigiert hat.

In [ ]:
from IPython.display import Markdown

random.seed(4)
radar.reset_verbrauch()
probe = df.sample(20, random_state=1)
ergebnisse = [analysiere(t, stoerung=0.3) for t in probe["text"]]

# 1. Überblick
def ergebnis_text(a, n):
    if not a:
        return "❌ aufgegeben"
    return "✅ sofort gültig" if n == 1 else f"✅ gültig nach {n - 1} Korrektur(en)"

display(Markdown("**1. Überblick: 20 Bewertungen**"))
uebersicht = pd.DataFrame([{"Bewertung": t[:60], "Störung": p["stoerung"] or "—",
                            "Anfragen": n, "Ergebnis": ergebnis_text(a, n)}
                           for t, (a, n, p) in zip(probe["text"], ergebnisse)],
                          index=range(1, len(probe) + 1))
display(uebersicht)

# 2. Bilanz
gestoert = sum(1 for _, _, p in ergebnisse if p["stoerung"])
korrekturen = sum(n - 1 for _, n, _ in ergebnisse)
display(Markdown("**2. Bilanz**"))
print(f"Gestört: {gestoert} von {len(ergebnisse)} Antworten (eingestellt: 30 %)")
print(f"Anfragen: {len(ergebnisse)} Erstanfragen + {korrekturen} Korrektur-Anfragen = {len(ergebnisse) + korrekturen}")
print("Verbrauch laut Zähler:", radar.verbrauch())

# 3. Ein Fall im Detail (FALL = 0, 1, 2 ... für den nächsten gestörten Fall)
FALL = 0
gestoerte = [(t, a, n, p) for t, (a, n, p) in zip(probe["text"], ergebnisse) if p["stoerung"]]
if gestoerte:
    t, a, n, p = gestoerte[FALL]
    display(Markdown(f"**3. Ein Fall im Detail** (gestörter Fall {FALL + 1} von {len(gestoerte)})"))
    print(f"Bewertung:\n   {t}\n")
    print(f"① Originalantwort des Modells:\n   {p['original']}\n")
    print(f"② Unsere Störung „{p['stoerung']}\" macht daraus:\n   {p['schritte'][0]['antwort']}\n")
    print(f"③ Prüfer meldet:\n   {p['schritte'][0]['fehler']}\n")
    print("④ Das schicken wir dem Modell zurück:")
    print("   " + p["schritte"][0]["gesendet"].replace("\n", "\n   ") + "\n")
    for i, s in enumerate(p["schritte"][1:], start=1):
        print(f"⑤ Korrigierte Antwort ({i}. Korrektur):\n   {s['antwort']}")
        print(f"   Prüfer: {'✅ gültig' if not s['fehler'] else '❌ ' + s['fehler']}\n")

**Wie lese ich das?**
- **Überblick:** Etwa jede dritte Antwort wurde gestört. Die Spalte „Anfragen" zeigt den Preis: 1 = sofort gültig, 2 = eine Korrektur nötig.
- **Bilanz:** Jede Korrektur ist eine **zusätzliche Anfrage**. Die Störung trifft nur die erste Antwort, deshalb braucht ein gestörter Fall normalerweise genau eine Korrektur. Wird ein Fall zweimal korrigiert, hat das Modell auch bei der Korrektur einen echten Fehler gemacht.
- **30 % ist eine Wahrscheinlichkeit, keine Quote:** Für jede Bewertung wird einzeln „gewürfelt". Bei 20 Bewertungen sind im Mittel 6 gestört, aber 4 bis 8 sind völlig normal, wie bei kleinen Stichproben in Sitzung 3.
- **Der Fall im Detail:** ① war eigentlich in Ordnung, ② ist unsere Störung. Wichtig ist ④: Das Modell bekommt die Bewertung, seine (gestörte) Antwort **und die Fehlermeldung des Prüfers**. Es muss nicht raten, was falsch war. In ⑤ repariert es gezielt genau diese Stelle.
- Setzt `FALL = 1`, `2`, … und schaut euch die anderen Störungsarten an. Besonders interessant: „abgeschnitten". Da muss das Modell die Antwort neu schreiben.

Was „aufgegeben" wird, bekommt **kein geratenes Label**, sondern bleibt leer und wird gezählt, wie die Quarantäne in Sitzung 4.

So sieht ein gültiges Ergebnis als Tabelle aus, mit Feldern, die ein Dashboard direkt verwenden kann:

In [ ]:
tabelle = pd.DataFrame([{"Bewertung": t, "sentiment": a.sentiment, "dringend": "⚠️ ja" if a.dringend else "nein",
                         "themen": ", ".join(f"{n.thema} ({n.polaritaet})" for n in a.themen),
                         "Begründung des Modells": a.begruendung, "Anfragen": n}
                        for t, (a, n, _) in zip(probe["text"], ergebnisse) if a])
with pd.option_context("display.max_colwidth", None):
    display(tabelle.head(10))

**Wie lese ich das?** Jede Zeile hat jetzt Felder, die ein Dashboard direkt verwenden kann: filtern nach `dringend`, zählen nach Thema, sortieren nach Sentiment.

Aber schaut genau auf `dringend`. Unser Prompt definiert es mit einem einzigen Satz: *„true, wenn ein Defekt oder ein Problem gemeldet wird, um das sich jemand kümmern sollte"*. Was „kümmern" heißt, entscheidet das Modell. Sucht euch zwei Bewertungen mit derselben Beschwerde (z. B. „Akku nach einer Stunde leer") und vergleicht: Ist `dringend` gleich? Was steht in der Begründung?

- Die **Begründung** ist das einzige Feld, das zeigt, *warum* das Modell so entschieden hat. Ohne sie sähen wir nur ein `True` oder `False`.
- Das Schema garantiert: `dringend` ist wahr oder falsch. Es garantiert **nicht**, dass es richtig ist (Fall 11 aus Abschnitt 1).
- Ein unscharf definiertes Feld liefert unscharfe Werte, genau wie die Themen in Sitzung 5 ohne Katalog und Definitionen.

> **Nicht:** „Das Modell weiß schon, was dringend ist."
> **Sondern:** Was dringend ist, entscheidet das **Produktteam**, und schreibt es so genau in den Prompt, dass es messbar wird.

## 4. Das Schema direkt an den Anbieter geben

Prüfen und korrigieren funktioniert mit jedem Modell. Viele Anbieter bieten aber auch an, das Schema **direkt mitzuschicken**. Bei Anthropic heißt das *Tool Use*: Man beschreibt ein „Werkzeug" mit seinen Eingabefeldern, und das Modell füllt diese Felder aus. Mit der Einstellung `strict: true` **garantiert** der Anbieter, dass die Antwort zum Schema passt. Ohne `strict` hält sich das Modell meistens daran, aber nicht immer.

Praktisch: pydantic kann aus unserer Klasse das passende JSON-Schema erzeugen. **Was passiert in der nächsten Zelle?** Wir erzeugen das Schema und passen es für den strict-Modus an.

In [ ]:
def ohne_verweise(schema):
    """pydantic legt Unter-Objekte unter "$defs" ab und verweist mit "$ref" darauf.
    Wir setzen die Verweise direkt ein, das ist für alle Anbieter am verträglichsten."""
    defs = schema.get("$defs", {})
    def ersetze(x):
        if isinstance(x, dict):
            if "$ref" in x:
                return ersetze(defs[x["$ref"].split("/")[-1]])
            return {k: ersetze(v) for k, v in x.items() if k != "$defs"}
        if isinstance(x, list):
            return [ersetze(v) for v in x]
        return x
    return ersetze(schema)

def fuer_strict(schema):
    """Der strict-Modus unterstützt keine Längenregeln (maxLength) und verlangt
    additionalProperties: false bei jedem Objekt (keine zusätzlichen Felder)."""
    if isinstance(schema, dict):
        neu = {k: fuer_strict(v) for k, v in schema.items() if k != "maxLength"}
        if neu.get("type") == "object":
            neu["additionalProperties"] = False
        return neu
    if isinstance(schema, list):
        return [fuer_strict(v) for v in schema]
    return schema

schema = fuer_strict(ohne_verweise(Analyse.model_json_schema()))
print(json.dumps(schema, indent=2, ensure_ascii=False)[:1500], "...")

**Wie lese ich das?** Das ist dieselbe Information wie in unserer Python-Klasse, nur in einem Standardformat (*JSON Schema*), das auch andere Programme verstehen: Feldnamen, Typen, erlaubte Werte (`enum`), Pflichtfelder (`required`). Zwei Unterschiede zur Klasse: `additionalProperties: false` verbietet zusätzliche Felder, und die Regel „`begruendung` höchstens 200 Zeichen" fehlt. Der strict-Modus unterstützt sie nicht, mitgeschickt gäbe es einen Fehler.

**Was passiert in der nächsten Zelle?** Wir schicken acht Bewertungen mit diesem Schema an das Modell. `tool_choice` zwingt das Modell, das Werkzeug zu benutzen, statt frei zu antworten. Die Antwort kommt dann nicht als Text, sondern als fertiges Python-Dictionary. Danach prüfen wir mit unserer **eigenen** Klasse `Analyse`, die die 200-Zeichen-Regel noch kennt.

In [ ]:
def analysiere_mit_schema(text):
    import anthropic
    client = anthropic.Anthropic(api_key=radar.get_secret("ANTHROPIC_API_KEY"))
    r = client.messages.create(
        model=radar.MODEL, max_tokens=400,
        system="Du analysierst Produktbewertungen zu kabellosen Ohrhörern.",
        tools=[{"name": "analyse_speichern",
                "description": "Speichert die Analyse einer Bewertung.",
                "input_schema": schema,
                "strict": True}],                                         # Anbieter garantiert das Schema
        tool_choice={"type": "tool", "name": "analyse_speichern"},     # Werkzeug ist Pflicht
        messages=[{"role": "user", "content": f'Bewertung:\n"""{text}"""'}])
    return next(b.input for b in r.content if b.type == "tool_use")    # schon ein Dictionary

if radar.BACKEND == "anthropic":
    for t in probe["text"].head(8):
        daten = analysiere_mit_schema(t)
        try:
            analyse = Analyse.model_validate(daten)
            print(f"✅ {t[:60]!r}\n   {analyse.sentiment}, dringend={analyse.dringend}, "
                  f"{[(n.thema, n.polaritaet) for n in analyse.themen]}\n")
        except ValidationError as e:
            f = e.errors()[0]
            print(f"❌ {t[:60]!r}\n   Schema verletzt: {f['loc'][0]}: {f['msg']}")
            print(f"   ({len(daten.get('begruendung', ''))} Zeichen: {daten.get('begruendung', '')[:80]!r} ...)\n")
else:
    print("Ohne Schlüssel (Mock) nicht verfügbar. Diese Zelle braucht das echte Modell.")

**Wie lese ich das?** Kein Herausschneiden, kein Reparieren: Die Antwort kommt direkt als Dictionary, und dank `strict` garantiert mit den richtigen Feldern und erlaubten Werten.

Aber der Anbieter setzt nur durch, was im Schema steht, und die 200-Zeichen-Regel konnten wir gar nicht mitschicken. Erscheint oben ein ❌, ist das genau dieser Fall: Unsere eigene Prüfung mit `Analyse.model_validate` fängt die zu lange Begründung ab. Ohne sie stünde der Text jetzt in der Datenbank. Sind alle ✅, hatten wir diesmal Glück, garantiert ist es nicht.

Warum also nicht blind dem Anbieter-Schema vertrauen?
- Nicht jede Regel lässt sich beim Anbieter durchsetzen. Welche, unterscheidet sich je nach Anbieter und Modell.
- Auch eine formal perfekte Antwort kann inhaltlich falsch sein: Ein gültiges `"sentiment": "positive"` bei einer sarkastischen Bewertung erfüllt jedes Schema (Fall 11).

> **Nicht:** „Mit Schema beim Anbieter brauchen wir keine eigene Prüfung mehr."
> **Sondern:** Der Anbieter garantiert die **Form**. Zusätzliche Regeln und den **Inhalt** prüfen wir trotzdem selbst, und bei Fehlern greift die Korrektur aus Abschnitt 3.

## 5. Embeddings, Teil 2: Themen finden, die niemand kannte

Bisher haben **wir** die Themen festgelegt (Katalog). Was, wenn es ein Thema gibt, an das niemand gedacht hat? Dann hilft die Landkarte aus Sitzung 5: Bewertungen mit ähnlicher Bedeutung liegen nah beieinander. Ein Verfahren, das **Gruppen von nahen Punkten** findet, heißt *Clustering*.

Das bekannteste ist **k-Means**: Man gibt vor, wie viele Gruppen es geben soll (k), und der Algorithmus teilt die Punkte so auf, dass jede Gruppe möglichst eng beieinander liegt.

**Was passiert in der nächsten Zelle?** Wir übersetzen alle Bewertungen in Embeddings (wie letzte Woche), lassen k-Means 10 Gruppen bilden und zeichnen die Landkarte, diesmal gefärbt nach **gefundener Gruppe** statt nach wahrem Thema.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

E = radar.embed(df["text"])
K = 10
df["cluster"] = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(E)
xy = PCA(n_components=2, random_state=0).fit_transform(E)

fig, ax = plt.subplots(figsize=(8, 6))
farben = plt.cm.tab20.colors                       # 20 Farben, reicht auch für k = 12
for c in range(K):
    m = (df["cluster"] == c).values
    ax.scatter(xy[m, 0], xy[m, 1], s=14, color=farben[c % 20], label=f"Gruppe {c} ({m.sum()})")
ax.legend(fontsize=8, frameon=False, bbox_to_anchor=(1, 1), loc="upper left")
ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"Gefundene Gruppen (k-Means, k = {K})")
plt.tight_layout(); plt.show()

**Wie lese ich das?** Jede Farbe ist eine Gruppe, die k-Means gefunden hat, **ohne** unseren Katalog zu kennen. Dass sich Farben auf der Karte überlappen, ist kein Fehler: k-Means gruppiert im Raum mit 384 Richtungen, die Karte zeigt davon nur die zwei wichtigsten. Punkte, die hier übereinander liegen, können dort weit auseinander sein.

Aber: k-Means weiß nicht, **was** eine Gruppe bedeutet. Es liefert nur Nummern. Um das herauszufinden, schauen wir hinein.

**Was passiert in der nächsten Zelle?** Für jede Gruppe drei Beispiel-Bewertungen.

In [ ]:
for c in range(K):
    beispiele = df.loc[df["cluster"] == c, "text"].drop_duplicates().head(3).tolist()   # jeden Text nur einmal
    print(f"Gruppe {c} ({(df['cluster'] == c).sum()} Bewertungen)")
    for b in beispiele:
        print("   ", b[:85])
    print()

**Wie lese ich das?** Lest die Beispiele einer Gruppe und überlegt: Wie würdet ihr sie nennen? Manche Gruppen sind klar ein Thema („alles zum Akku"). Bei anderen lohnt die Frage, ob k-Means nach etwas anderem sortiert hat: nach **Stimmung** (nur Lob), nach **Satzmuster** („Habe sie seit Kurzem …") oder nach einem Muster quer zu den Themen (Defekte).

Das Benennen kann auch das LLM übernehmen. **Was passiert in der nächsten Zelle?** Eine einzige Anfrage mit je fünf Beispielen pro Gruppe, das Modell schlägt Namen vor.

In [ ]:
beispiele = "\n\n".join(f"Gruppe {c}:\n" + "\n".join("- " + t for t in df.loc[df["cluster"] == c, "text"].drop_duplicates().head(5))
                         for c in range(K))
anfrage = ("Hier sind Gruppen ähnlicher Produktbewertungen zu Ohrhörern. Gib jeder Gruppe einen kurzen Namen "
           "(2-4 Wörter), der beschreibt, worum es geht. Antworte nur als JSON-Objekt "
           '{"0": "<Name>", "1": "<Name>", ...}.\n\n' + beispiele)

namen = {str(c): f"Gruppe {c}" for c in range(K)}             # Ersatz, falls etwas schiefgeht
if radar.BACKEND == "anthropic":
    antwort = radar.call_llm(anfrage, max_tokens=400)["text"]
    try:                                                        # was wir heute gelernt haben: nie blind json.loads
        namen.update(json.loads(re.search(r"\{.*\}", antwort, re.DOTALL).group(0)))
    except (AttributeError, json.JSONDecodeError):
        print("⚠️ Antwort nicht lesbar, nutze Nummern:", antwort[:100])
else:
    print("Mock: keine echten Namen.")
for c in range(K):
    print(f"Gruppe {c}: {namen.get(str(c))}")

**Wie lese ich das?** Das sind **Vorschläge**, keine Wahrheit. Sie hängen davon ab, welche Beispiele wir mitgeschickt haben.

**Wie passen die gefundenen Gruppen zu unserem Katalog?** Die nächste Zelle zeigt für jede Gruppe, welche wahren Themen darin vorkommen. Je dunkler ein Feld, desto mehr Bewertungen der Gruppe haben dieses Thema.

In [ ]:
themen_wahr = [{t for t, _ in radar.wahre_themen(wahrheit.loc[r, "text_wahr"])} for r in df["review_id"]]
matrix = pd.DataFrame([[sum(th in t for t, c in zip(themen_wahr, df["cluster"]) if c == k) for th in THEMEN]
                       for k in range(K)], columns=THEMEN,
                      index=[f"{k}: {namen.get(str(k))}" for k in range(K)])
anteil = matrix.div(df["cluster"].value_counts().sort_index().values, axis=0)

fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(anteil.values, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(THEMEN))); ax.set_xticklabels(THEMEN, rotation=40, ha="right")
ax.set_yticks(range(K)); ax.set_yticklabels(anteil.index)
for i in range(K):
    for j in range(len(THEMEN)):
        ax.text(j, i, f"{anteil.values[i, j]:.0%}", ha="center", va="center", fontsize=7,
                color="white" if anteil.values[i, j] > 0.6 else "#111827")
ax.set_title("Welche Themen stecken in welcher Gruppe?")
plt.tight_layout(); plt.show()

**Wie lese ich das?** Eine Zeile pro gefundener Gruppe. Steht in einer Zeile ein dunkles Feld, ist die Gruppe im Wesentlichen **ein** Thema: Clustering und Katalog stimmen überein. Verteilt sich eine Zeile auf viele helle Felder, hat k-Means nach etwas anderem gruppiert (Satzbau, Sprache, Stimmung).

### Welches k? Gemeinsam ausprobieren

k-Means braucht von uns die Zahl der Gruppen. Zu klein: verschiedene Themen werden in einen Topf geworfen. Zu groß: ein Thema zerfällt in viele Mini-Gruppen. Zwei Kennzahlen helfen bei der Wahl:

- **Silhouette:** Wie gut sind die Gruppen voneinander getrennt? Höher ist besser. Die Werte sind bei Texten oft klein; wichtig ist der **Vergleich** zwischen den k.
- **Themen-Reinheit:** Wie stark wird jede Gruppe von **einem** Katalog-Thema bestimmt? 100 % hieße: Jede Gruppe ist genau ein Thema. Diese Kennzahl gibt es nur, weil wir hier die Wahrheit kennen. In echten Daten hätte man nur die Silhouette und den Blick in die Gruppen.

**Was passiert in der nächsten Zelle?** k-Means für k = 3 bis 12, für jedes k beide Kennzahlen.

In [ ]:
from sklearn.metrics import silhouette_score

def themen_reinheit(labels, k):
    """Gewichteter Anteil des häufigsten Katalog-Themas je Gruppe."""
    gesamt = 0
    for c in range(k):
        in_c = [t for t, l in zip(themen_wahr, labels) if l == c]
        if in_c:
            gesamt += max(sum(th in t for t in in_c) for th in THEMEN)
    return gesamt / len(labels)

zeilen = []
for k in [3, 4, 5, 6, 8, 10, 12]:
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(E)
    zeilen.append({"k": k, "Silhouette": silhouette_score(E, labels, metric="cosine"),
                   "Themen-Reinheit": themen_reinheit(labels, k)})
vergleich = pd.DataFrame(zeilen).set_index("k")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, spalte, farbe in zip(axes, vergleich.columns, ["#2563eb", "#059669"]):
    ax.plot(vergleich.index, vergleich[spalte], marker="o", color=farbe)
    ax.set_title(spalte); ax.set_xlabel("k (Anzahl Gruppen)"); ax.set_xticks(vergleich.index)
    ax.spines[["top", "right"]].set_visible(False)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.tight_layout(); plt.show()
vergleich.style.format({"Silhouette": "{:.3f}", "Themen-Reinheit": "{:.0%}"})

**Wie lese ich das?** Sucht ein k, bei dem die Silhouette gut ist **und** die Reinheit nicht weiter stark steigt. Mehr Gruppen machen die Reinheit fast immer etwas besser, aber irgendwann zerfallen nur noch Themen in Teile. Oben haben wir schon `K = 10` gesetzt. Probiert gern andere Werte (z. B. 4 oder 12) und führt die Zellen ab der Landkarte erneut aus. Die Auswertung unten gilt aber für k = 10.

**Was passiert in der nächsten Zelle?** Für das gewählte k ein Steckbrief je Gruppe: Hauptthema, überwiegende Stimmung, Beispiel. Die letzte Spalte ist eine einfache Vermutung, wonach k-Means sortiert hat.

In [ ]:
sent_wahr = wahrheit.loc[df["review_id"], "true_sentiment"].values
REIHENFOLGE = ["positive", "mixed", "neutral", "negative"]

profil = []
for c in range(K):
    m = (df["cluster"] == c).values
    th_anteile = anteil.iloc[c]
    verteilung = pd.Series(sent_wahr[m]).value_counts(normalize=True)
    mitte = E[m].mean(axis=0)                                  # Mittelpunkt der Gruppe
    beispiel = df.loc[m, "text"].iloc[np.argmax(E[m] @ mitte)] # Bewertung, die der Mitte am nächsten liegt
    if th_anteile.max() >= 0.7:
        vermutung = "Thema"
    elif verteilung.max() >= 0.8:
        vermutung = "Stimmung"
    else:
        vermutung = "gemischt"
    profil.append({"Gruppe": f"{c}: {namen.get(str(c))}", "Größe": m.sum(),
                   "Hauptthema (Anteil)": f"{th_anteile.idxmax()} ({th_anteile.max():.0%})",
                   "Stimmung (wahr)": " · ".join(f"{s} {verteilung.get(s, 0):.0%}" for s in REIHENFOLGE
                                                 if verteilung.get(s, 0) > 0),
                   "typisches Beispiel": beispiel, "sortiert nach (Faustregel)": vermutung})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(profil).set_index("Gruppe"))

### Was sagt uns die Clusteranalyse?

Die Faustregel im Steckbrief sortiert jede Gruppe automatisch in eine von drei Schubladen. Danach schaut ein Mensch genauer hin, denn manche Muster sieht man nur im **Vergleich der Gruppen** oder beim **Lesen der Beispiele**:

| Faustregel sagt | Worauf wir dann schauen | Mögliches Ergebnis | Entscheidung |
|---|---|---|---|
| Thema | Spaltensicht: Liegt dieses Thema auch in einer anderen Gruppe? | Nein: Clustering findet dasselbe wie unser Katalog. Ja: Das Thema zerfällt in Unterthemen | Katalog bestätigt, oder prüfen: Thema aufteilen? |
| Stimmung | Beispiele: Lob oder Beschwerden quer über viele Themen? Geht es um Defekte? | Sortiert nach Ton, nicht nach Inhalt. Bei Defekten: ein Muster quer zu den Themen | Kein Thema. Defekte sind genau das, was `dringend` erfassen soll |
| gemischt | Beispiele: Was verbindet diese Bewertungen? | Ein Satzmuster, ein Defekt, oder etwas, das in keinem Thema vorkommt | Kein Thema, ein Merkmal, oder ein **Kandidat für ein neues Thema** |

Unser Ergebnis (gilt für `K = 10`; die Gruppennummern bleiben bei jedem Lauf gleich, die Namen des LLM nicht):

```text
Ergebnis der Clusteranalyse (k = 10)
- Warum 10:         Reinheit am höchsten (78 %), bei 12 nicht mehr besser: danach zerfallen nur noch Themen
- Bestätigt:        7 von 10 Gruppen sind im Wesentlichen ein Katalog-Thema
                    (Tragekomfort, App & Bedienung, Geräuschunterdrückung, Verbindung,
                     Akku & Laden, Klang, Preis)
- Anders sortiert:  Gruppe 1 = Beschwerden quer über Themen (89 % negativ),
                    Gruppe 7 = Lob quer über Themen (83 % positiv): sortiert nach Ton, nicht Inhalt
- Auffällig:        "Akku & Laden" verteilt sich: Gruppe 6 = Laufzeit, Gruppe 4 = "lädt nicht mehr"
                    (ein Defekt, keine Laufzeitfrage)
Entscheidung:       Katalog bleibt, kein neues Thema gefunden.
                    Diskutieren: "Akku & Laden" in Laufzeit und Lade-Defekt aufteilen?
                    Defekte quer über Themen sind kein Thema, sondern ein Merkmal:
                    genau dafür gibt es "dringend", und das muss schärfer definiert werden.
```

**Und die Namen?** Das LLM hat aus fünf Beispielen pro Gruppe benannt. Vergleicht Namen und Steckbrief: Wie heißt Gruppe 1 bei euch? Sie ist nur zur Hälfte ein Verbindungsthema. Und Gruppe 6: Klingt ihr Name nach Problemen, obwohl weniger als die Hälfte negativ ist? Namen sind **Vorschläge**, geprüft wird mit den Zahlen, wie bei der Begründung und bei `dringend`.

**Warum kein neues Thema?** Unsere Daten sind künstlich erzeugt, jede Bewertung besteht aus Bausteinen des Katalogs. In echten Daten wäre genau hier der Moment, ein Thema zu entdecken, an das niemand gedacht hat (z. B. Lieferung oder Kundenservice).

> 💡 **Kernpunkt:** Clustering **findet** Gruppen, aber es **benennt** sie nicht, und wie viele es gibt, entscheidet ihr (k).
>
> **Nicht:** „Die Daten sagen uns, welche Themen es gibt."
> **Sondern:** Clustering macht **Vorschläge**. Ein Mensch schaut hinein, benennt, und entscheidet, ob daraus ein neues Thema im Katalog wird. Zum Zählen nehmen wir weiter den festen Katalog.

## Zusammenfassung

```text
1. Modelle liefern kein sicheres     -> nie blind json.loads; erst herausschneiden, dann prüfen
   Format
2. Gültiges JSON != richtige         -> Schema als Vertrag (pydantic): Felder, Typen, erlaubte Werte
   Antwort
3. Schema prüft die Form,            -> Begründung lesen: zeigt, warum das Modell so entschieden hat
   nicht den Sinn                       (Fall 11, unscharfes "dringend")
4. Antwort ungültig                  -> Fehlermeldung ans Modell zurück, höchstens 2 Korrekturen,
                                        sonst leer lassen und zählen
5. Schema beim Anbieter (strict)     -> Form garantiert; Zusatzregeln und Inhalt prüfen wir selbst
6. Unbekannte Themen                 -> Clustering macht Vorschläge, ein Mensch benennt und entscheidet
```

## Zum Weiterprobieren (freiwillig)

1. **Batch mit Schema:** Erweitert das Schema auf eine Liste von Analysen mit `id` und schickt 20 Bewertungen auf einmal (Sitzung 3). Wie oft fehlt jetzt eine ID?
2. **Störung hochdrehen:** Setzt `stoerung=1.0`. Wie viele Antworten werden nach einer, nach zwei Korrekturen gültig? Was kostet das an zusätzlichen Aufrufen?
3. **Dringend messen:** Wie gut erkennt das Modell „dringend"? Legt selbst für 20 Bewertungen fest, ob sie dringend sind, und vergleicht. Schärft danach die Definition im Prompt und schaut, ob die Werte einheitlicher werden.

## Ausblick

**Nächste Woche (18.11.): Aggregation und Trends.** Jetzt haben wir für jede Bewertung saubere Felder: Sentiment, Themen, dringend. Nächste Woche verdichten wir sie: Wie entwickeln sich die Themen über die Zeit, und welche Fragen der Produktmanagerin können wir damit beantworten?